# A2.5 · The non-human identity lifecycle

**Function A — Securing AI Architectures → Securing the Architecture — Identity and Ingress**  ·  *Security of AI*

Builds on **[A2.4 · Just-in-time authority](https://spbreed.github.io/cyber-commons/lessons/A2.4.html)**.

| | |
|---|---|
| Tools used | SCIM 2.0 (RFC 7643/7644), Keycloak, SPIFFE/SPIRE |

## What this lesson is

**What it covers.** Admit agents against a registry and show an unregistered one refused at the door.

**Why a security engineer needs it.** Agents accumulate with no owner and no expiry, and an unregistered agent joins a topology as a peer. The control it builds is: a registry with a named owner, an expiry, and admission bound to a registered identity.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Non-human identities already outnumber humans in most estates, and they sit outside joiner-mover-leaver entirely. Nobody ever leaves, so nothing is ever revoked, and last year's proof-of-concept still holds production write. The protocol that fixes it is one you already run for humans.

> **At CyberTravels.** `cybertravels-svc` was created for a proof of concept in March. The proof of concept was cancelled. The identity still holds payments scope, because nothing in CyberTravels' joiner-mover-leaver process describes an agent.

## 2 · The framework

```
   humans                         non-human identities
   joiner -> mover -> leaver      created -> ... -> ?
      |                              |
   HR system drives it            nothing drives it
   revocation is automatic        nobody ever leaves

   the fix is to run ONE protocol over both:

     HR/IdP --SCIM--> POST   /Users     a person joins
                      POST   /Agents    an agent is deployed
                      PATCH  active:false   a leaver, or a retirement

   owner is a $ref to a User, not a string, so the leaver
   event that already exists is the one that retires the agent
```

**Mitigates: T9 Identity Spoofing · T13 Rogue Agents · T3 Privilege Compromise.**

Human identities have a lifecycle: someone joins, moves team, leaves, and an HR
event drives the change. Non-human identities have none of that. They are
created by whoever needed one, owned by nobody in particular, and removed never.

They also outnumber humans, often by a large multiple.

A1.11's rogue agent was admitted because the orchestrator had no notion of an
approved agent. The control is a registry, and a registry is only useful if it
carries three fields:

**A named owner.** A person, not a team alias. An identity with no owner cannot
be renewed, questioned or revoked, because nobody is accountable for answering.

**An expiry.** Not for the credential — for the *registration*. It forces a
recurring decision about whether this agent should still exist, which is the
only mechanism that removes the ones nobody uses.

**An admission binding.** The registry entry names the workload identity from
A2.2. Admission then checks the presented identity against the registry rather
than checking a name the caller supplied.

That last point is what makes it a control rather than a spreadsheet. A registry
consulted by name is documentation; a registry consulted by attested identity is
an authorization decision.

### The registry has to be driven by something, and that something is SCIM

A registry nobody updates decays into the spreadsheet it replaced. Humans do
not have this problem, because their lifecycle is already automated by a
protocol: **SCIM** — System for Cross-domain Identity Management, RFC 7643 for
the schema and RFC 7644 for the protocol. The HR system creates a user, the
identity provider `POST`s it to `/Users`, a leaver event `PATCH`es
`{"active": false}`, and every downstream application finds out without anybody
filing a ticket.

Point the same protocol at agents and three things become true at once:

**The joiner-mover-leaver machinery you already run is the machinery that
governs agents.** No parallel process, no second system of record. Deploying an
agent means creating a SCIM resource; retiring it means one `PATCH`.

**The owner is a reference, not a string.** `owner.$ref` points at the SCIM
`User` — so when Sam leaves, Sam's own leaver event is enough to answer "which
agents just lost their owner", automatically, on the day it happens rather than
at the next audit.

**Orphans become a query.** `GET /Agents?filter=active eq true and owner pr
false` is a one-line answer to a question that is otherwise a quarter of
someone's life.

> **What is and is not standard here.** SCIM's protocol — the endpoints, the
> filter syntax, `PATCH` with `active: false`, the `meta` block — is RFC 7644
> and your IdP already speaks it. A resource type for *agents* is not in RFC
> 7643; you declare it as a schema extension under your own URN, exactly as the
> enterprise extension does for `manager`. The protocol is standard, the schema
> is yours, and that split is the whole reason this is cheap to adopt.

> **What this control closes.**
>
> Turns 'which agents are allowed here' from a convention into a check, and turns 'is this one still wanted' from an audit question into a leaver event. Closes A1.11 at the door.

<div style="display:flex;gap:6px;align-items:stretch;flex-wrap:wrap;font-family:ui-sans-serif,system-ui,-apple-system,Segoe UI,Roboto,sans-serif;margin:6px 0 2px"><div style="flex:1 1 150px;min-width:150px"><div style="font-size:10px;text-transform:uppercase;letter-spacing:.09em;color:#8A93A6;font-weight:600;padding-bottom:5px;border-bottom:1px solid rgba(138,147,166,.4);margin-bottom:3px">system of record</div><div style="border:1px solid rgba(63,160,107,.55);border-left:3px solid #3FA06B;border-radius:8px;background:rgba(63,160,107,.10);padding:8px 11px;margin:5px 0;min-width:132px"><div style="font-size:13px;font-weight:600"><span style="font-size:15px">&#127970;</span> HR / IdP</div><div style="font-size:11px;color:#8A93A6;margin-top:2px;line-height:1.35">the one place a joiner, mover or leaver is recorded</div></div></div><div style="align-self:center;color:rgba(138,147,166,.8);font-size:20px;padding:0 2px;flex:0 0 auto">&#8250;</div><div style="flex:1 1 150px;min-width:150px"><div style="font-size:10px;text-transform:uppercase;letter-spacing:.09em;color:#8A93A6;font-weight:600;padding-bottom:5px;border-bottom:1px solid rgba(138,147,166,.4);margin-bottom:3px">scim protocol</div><div style="border:1px solid rgba(224,145,47,.55);border-left:3px solid #E0912F;border-radius:8px;background:rgba(224,145,47,.10);padding:8px 11px;margin:5px 0;min-width:132px"><div style="font-size:13px;font-weight:600"><span style="font-size:15px">&#128100;</span> POST /Users</div><div style="font-size:11px;color:#8A93A6;margin-top:2px;line-height:1.35">a person joins</div></div><div style="border:1px solid rgba(77,155,255,.55);border-left:3px solid #4D9BFF;border-radius:8px;background:rgba(77,155,255,.10);padding:8px 11px;margin:5px 0;min-width:132px"><div style="font-size:13px;font-weight:600"><span style="font-size:15px">&#129302;</span> POST /Agents</div><div style="font-size:11px;color:#8A93A6;margin-top:2px;line-height:1.35">an agent is deployed — same protocol, your own schema URN</div></div><div style="border:1px solid rgba(224,145,47,.55);border-left:3px solid #E0912F;border-radius:8px;background:rgba(224,145,47,.10);padding:8px 11px;margin:5px 0;min-width:132px"><div style="font-size:13px;font-weight:600"><span style="font-size:15px">&#9940;</span> PATCH active:false</div><div style="font-size:11px;color:#8A93A6;margin-top:2px;line-height:1.35">a leaver, or a retirement. One call, and every consumer finds out</div><div style="font-size:10px;color:#E0912F;margin-top:5px;font-weight:600;letter-spacing:.02em">NOT DELETE — THE RECORD SURVIVES</div></div></div><div style="align-self:center;color:rgba(138,147,166,.8);font-size:20px;padding:0 2px;flex:0 0 auto">&#8250;</div><div style="flex:1 1 150px;min-width:150px"><div style="font-size:10px;text-transform:uppercase;letter-spacing:.09em;color:#8A93A6;font-weight:600;padding-bottom:5px;border-bottom:1px solid rgba(138,147,166,.4);margin-bottom:3px">agent registry</div><div style="border:1px solid rgba(77,155,255,.55);border-left:3px solid #4D9BFF;border-radius:8px;background:rgba(77,155,255,.10);padding:8px 11px;margin:5px 0;min-width:132px"><div style="font-size:13px;font-weight:600"><span style="font-size:15px">&#128220;</span> the registry</div><div style="font-size:11px;color:#8A93A6;margin-top:2px;line-height:1.35">owner is a $ref to a User, not a string. Expiry is on the registration, not the credential</div></div></div><div style="align-self:center;color:rgba(138,147,166,.8);font-size:20px;padding:0 2px;flex:0 0 auto">&#8250;</div><div style="flex:1 1 150px;min-width:150px"><div style="font-size:10px;text-transform:uppercase;letter-spacing:.09em;color:#8A93A6;font-weight:600;padding-bottom:5px;border-bottom:1px solid rgba(138,147,166,.4);margin-bottom:3px">admission</div><div style="border:1px solid rgba(224,92,75,.55);border-left:3px solid #E05C4B;border-radius:8px;background:rgba(224,92,75,.10);padding:8px 11px;margin:5px 0;min-width:132px"><div style="font-size:13px;font-weight:600"><span style="font-size:15px">&#128737;&#65039;</span> the orchestrator</div><div style="font-size:11px;color:#8A93A6;margin-top:2px;line-height:1.35">checks the attested SPIFFE ID against the registry, and resolves the owner ref before it admits anything</div><div style="font-size:10px;color:#E05C4B;margin-top:5px;font-weight:600;letter-spacing:.02em">R2</div></div></div></div><div style="font-size:12px;color:#8A93A6;margin-top:8px;line-height:1.5">The point of using SCIM rather than a table is the middle column: the leaver event that already exists is the one that retires the agent, and the owner reference is what makes it cascade.</div>

## 3 · Sam leaves. One SCIM `PATCH`, and what it does not reach

This is the part a registry without a protocol behind it gets wrong. Sam's leaver event fires correctly — his own account is deactivated the same afternoon. The agent he deployed keeps running.

## Your turn

Count your non-human identities and how many have a named human owner that resolves to a live account. The difference is the set nobody can revoke during an incident, because nobody can be asked whether it is still needed. Then check whether your IdP's SCIM connector can carry a custom resource type — most can, and it is usually a configuration rather than a project.

---

**Next → [A2.6 · Ingress: marking untrusted content at the door](https://spbreed.github.io/cyber-commons/lessons/A2.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*